In [167]:
#Installing and Importing yfinance

# Import pandas for data manipulation and time-series analysis
import pandas as pd
# Import numpy for numerical and vectorized calculations
import numpy as np
# Import yfinance to pull historical market data from Yahoo Finance
import yfinance as yf
# Import statsmodels to run the Ordinary Least Squares (OLS) CAPM regression
import statsmodels.api as sm
# Import datetime to handle date arithmetic
from datetime import datetime
# Import dateutil.relativedelta to easily subtract exactly 2 years from today
from dateutil.relativedelta import relativedelta

In [52]:
#Fetching Historical Data for a Single Ticker
ticker = yf.Ticker("002594.SZ")
data = ticker.history(period="5y")
print(data)

                                Open       High        Low      Close  \
Date                                                                    
2021-09-13 00:00:00+08:00  86.582049  87.759795  83.629615  84.333038   
2021-09-14 00:00:00+08:00  84.342717  89.056929  82.945561  86.872452   
2021-09-15 00:00:00+08:00  86.440077  87.120909  83.894206  84.265282   
2021-09-16 00:00:00+08:00  84.265279  85.433343  80.838523  80.990173   
2021-09-17 00:00:00+08:00  81.635521  82.926196  80.825616  82.319580   
...                              ...        ...        ...        ...   
2026-09-07 00:00:00+08:00  86.889999  87.339996  86.199997  86.510002   
2026-09-08 00:00:00+08:00  86.389999  86.699997  85.430000  86.570000   
2026-09-09 00:00:00+08:00  86.570000  86.580002  85.500000  85.879997   
2026-09-10 00:00:00+08:00  85.110001  85.190002  83.480003  83.519997   
2026-09-11 00:00:00+08:00  83.169998  84.480003  82.660004  84.000000   

                              Volume  Dividends  S

In [50]:
#Downloading Data for Multiple Tickers
yf.download(["002594.SZ", "1211.HK"], start="2021-09-13", end="2026-09-11", group_by="ticker")

[*********************100%***********************]  2 of 2 completed


Ticker      002594.SZ                                                \
Price            Open       High        Low      Close       Volume   
Date                                                                  
2021-09-13  86.582049  87.759795  83.629615  84.333038   64047537.0   
2021-09-14  84.342710  89.056921  82.945553  86.872444  106407000.0   
2021-09-15  86.440069  87.120901  83.894198  84.265274   65642733.0   
2021-09-16  84.265279  85.433343  80.838523  80.990173   66242913.0   
2021-09-17  81.635514  82.926189  80.825609  82.319572   41826321.0   
...               ...        ...        ...        ...          ...   
2026-09-04  87.379997  88.550003  87.260002  87.400002   20343142.0   
2026-09-07  86.889999  87.339996  86.199997  86.510002   21926512.0   
2026-09-08  86.389999  86.699997  85.430000  86.570000   19726735.0   
2026-09-09  86.570000  86.580002  85.500000  85.879997   18185775.0   
2026-09-10  85.110001  85.190002  83.480003  83.519997   33073158.0   

Ticker        1211.HK                                               
Price            Open       High        Low      Close      Volume  
Date                                                                
2021-09-13  83.911950  84.233454  80.761229  82.304443  20808330.0  
2021-09-14  82.947433  85.969547  81.404227  84.554939  32589900.0  
2021-09-15  84.169131  85.390840  82.368727  83.076027  17369961.0  
2021-09-16  83.976244  83.976244  79.410924  80.053925  28114179.0  
2021-09-17  79.346615  82.754539  79.089417  82.754539  28978191.0  
...               ...        ...        ...        ...         ...  
2026-09-04  85.699997  86.900002  85.400002  86.150002  20988366.0  
2026-09-07  85.300003  85.750000  83.800003  84.500000  13868455.0  
2026-09-08  83.599998  84.400002  83.000000  83.849998  16813986.0  
2026-09-09  83.349998  83.500000  81.449997  81.800003  27604650.0  
2026-09-10  80.500000  80.849998  79.050003  79.550003  26256339.0  

[1264 rows x 10 columns]

In [55]:
#Accessing Company Information and Financials
info = ticker.info
info["sector"]

'Consumer Cyclical'

In [208]:
# Download historical market data

# Download adjusted closing prices for the asset and market
price_data = yf.download(
    [asset_ticker, market_ticker],
    start="2021-09-13",
    end="2026-09-11",
    auto_adjust=True
)["Close"]

# Download the 10-year US Treasury yield
rf_data = yf.download(
    rf_ticker,
    start="2021-09-13",
    end="2026-09-11",
    auto_adjust=False
)["Close"]

# Calculate daily returns
returns = price_data.pct_change()

# Convert Treasury yield from percentage to decimal
rf_data = rf_data / 100

# Combine asset/market returns and risk-free rate
data = returns.copy()
data["Risk_Free"] = rf_data


# Display the cleaned data
print("Cleaned data:")
display(data.head())
print("\nNumber of observations:", len(data))
print("\nRemaining NaN values:")
print(data.isna().sum())

[*********************100%***********************]  2 of 2 completed
[*********************100%***********************]  1 of 1 completed


Cleaned data:


Ticker,002594.SZ,^IXIC,Risk_Free
Date,,,
2021-09-13,NaN,NaN,NaN
2021-09-14,0.030112,-0.004490,NaN
2021-09-15,-0.030011,0.008231,NaN
2021-09-16,-0.038867,0.001345,NaN
2021-09-17,0.016414,-0.009086,NaN



Number of observations: 1298

Remaining NaN values:
Ticker
002594.SZ    118
^IXIC         89
Risk_Free    592
dtype: int64


In [205]:
# Create the main dataframe for the CAPM/SMA analysis

df = pd.DataFrame()

# Asset price
df['Asset_Price'] = price_data[asset_ticker]

# Market price
df['Market_Price'] = price_data[market_ticker]

# Calculate daily returns
df['Asset_Return'] = df['Asset_Price'].pct_change()
df['Market_Return'] = df['Market_Price'].pct_change()

# Add the risk-free rate
df['Risk_Free'] = rf_data

# Remove rows containing missing values
df = df.dropna()

# Display the first few rows
display(df.head())

# Check for remaining NaN values
print("Remaining NaN values:")
print(df.isna().sum())

,Asset_Price,Market_Price,Asset_Return,Market_Return,Risk_Free
Date,,,,,
2021-09-14,86.872444,15037.759766,0.030112,-0.004490,SOFR
2021-09-15,84.265274,15161.530273,-0.030011,0.008231,SOFR
2021-09-16,80.990173,15181.919922,-0.038867,0.001345,SOFR
2021-09-17,82.319572,15043.969727,0.016414,-0.009086,SOFR
2021-09-23,80.641685,15052.240234,0.011003,0.010431,SOFR


Remaining NaN values:
Asset_Price      0
Market_Price     0
Asset_Return     0
Market_Return    0
Risk_Free        0
dtype: int64


In [206]:
asset_ticker = '002594.SZ'
market_ticker = '^IXIC'

asset_data = yf.download(asset_ticker, start="2021-09-13", end="2026-09-11")
market_data = yf.download(market_ticker, start="2021-09-13", end="2026-09-11")

# Asset price
df['Asset_Price'] = asset_data['Close'].squeeze()
# Market price
df['Market_Price'] = market_data['Close'].squeeze()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [187]:
# -------------------------------
# CALCULATE DAILY RETURNS
# -------------------------------
stock_data['Stock_Return'] = stock_data['Close'].pct_change()
market_data['Market_Return'] = market_data['Close'].pct_change()

# Align dates
data = pd.concat([stock_data['Stock_Return'], market_data['Market_Return']], axis=1).dropna()

# -------------------------------
# CAPM: CALCULATE BETA
# -------------------------------
cov_matrix = np.cov(data['Stock_Return'], data['Market_Return'])
beta = cov_matrix[0, 1] / cov_matrix[1, 1]

# -------------------------------
# SMA CALCULATION
# -------------------------------
stock_data['SMA_20'] = stock_data['Close'].rolling(window=20).mean()
stock_data['SMA_50'] = stock_data['Close'].rolling(window=50).mean()

# -------------------------------
# FINAL DATAFRAME
# -------------------------------
final_df = stock_data[['Close', 'Stock_Return', 'SMA_20', 'SMA_50']].copy()
final_df['Market_Return'] = market_data['Market_Return']
final_df['Beta'] = beta  # Same beta for all rows (static CAPM beta)

# -------------------------------
# OUTPUT
# -------------------------------
print(f"CAPM Beta for {asset_ticker} vs {market_ticker}: {beta:.4f}")
print(final_df.tail(10))  # Show last 10 rows

# Save to CSV if needed
final_df.to_csv("capm_sma_analysis.csv", index=True)

CAPM Beta for 002594.SZ vs ^IXIC: 0.0976
Price           Close Stock_Return   SMA_20     SMA_50 Market_Return      Beta
Ticker      002594.SZ                                                         
Date                                                                          
2026-08-28  92.320000     0.009072  90.6570  89.276532     -0.005234  0.097575
2026-08-31  88.199997    -0.044627  90.3445  89.295251     -0.001194  0.097575
2026-09-01  88.709999     0.005782  90.2225  89.375777     -0.010281  0.097575
2026-09-02  86.800003    -0.021531  90.0215  89.451977      0.004523  0.097575
2026-09-03  87.309998     0.005876  89.9145  89.560295      0.013969  0.097575
2026-09-04  87.400002     0.001031  89.7825  89.750116     -0.002899  0.097575
2026-09-07  86.510002    -0.010183  89.5505  89.893443           NaN  0.097575
2026-09-08  86.570000     0.000694  89.3730  90.036775     -0.003229  0.097575
2026-09-09  85.879997    -0.007970  89.1465  90.147178     -0.006361  0.097575
2026-09-10 

C:\Users\alync\AppData\Local\Temp\ipykernel_26112\358511088.py:8: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  data = pd.concat([stock_data['Stock_Return'], market_data['Market_Return']], axis=1).dropna()


In [209]:
# ============================================================
# CREATE STRATEGIES FOR CAPM COMPARISON
# ============================================================

# ------------------------------------------------------------
# Strategy 1: Buy & Hold
# ------------------------------------------------------------

df['BuyHold_Return'] = df['Asset_Return']


# ------------------------------------------------------------
# Strategy 2: 50-Day SMA
# ------------------------------------------------------------

df['SMA_50'] = (
    df['Asset_Price']
    .rolling(window=50)
    .mean()
    .shift(1)
)
df['Signal_50'] = np.where(
    df['Asset_Price'].shift(1) > df['SMA_50'],
    1,
    0
)

df['SMA50_Return'] = (
    df['Signal_50'] * df['Asset_Return']
)


# ------------------------------------------------------------
# Strategy 3: 200-Day SMA
# ------------------------------------------------------------

df['SMA_200'] = (
    df['Asset_Price']
    .rolling(window=200)
    .mean()
    .shift(1)
)

df['Signal_200'] = np.where(
    df['Asset_Price'].shift(1) > df['SMA_200'],
    1,
    0
)

df['SMA200_Return'] = (
    df['Signal_200'] * df['Asset_Return']
)


# ------------------------------------------------------------
# Strategy 4: 50/200 SMA Crossover
# ------------------------------------------------------------
df['Crossover_Signal'] = np.where(
    df['SMA_50'] > df['SMA_200'],
    1,
    0
)

df['Crossover_Return'] = (
    df['Crossover_Signal'] * df['Asset_Return']
)


# Remove rows where the 200-day SMA is not available
df = df.dropna(
    subset=[
        'SMA_200',
        'SMA50_Return',
        'SMA200_Return',
        'Crossover_Return'
    ]
)

print("Strategies successfully created.")
display(
    df[
        [
            'Asset_Price',
            'Asset_Return',
            'BuyHold_Return',
            'SMA50_Return',
            'SMA200_Return',
            'Crossover_Return'
        ]
    ].head()
)

Strategies successfully created.


,Asset_Price,Asset_Return,BuyHold_Return,SMA50_Return,SMA200_Return,Crossover_Return
Date,,,,,,
2022-08-18,105.237274,-0.005763,-0.005763,-0.0,-0.005763,-0.005763
2022-08-19,103.090836,-0.020396,-0.020396,-0.0,-0.020396,-0.020396
2022-08-22,104.285095,0.011585,0.011585,0.0,0.011585,0.011585
2022-08-23,103.807388,-0.004581,-0.004581,-0.0,-0.004581,-0.004581
2022-08-24,99.898598,-0.037654,-0.037654,-0.0,-0.037654,-0.037654


In [222]:
# ============================================================
# CALCULATE EXCESS RETURNS
# ============================================================

df['Risk_Free'] = pd.to_numeric(df['Risk_Free'], errors='coerce')

df['BuyHold_Excess'] = (
    df['BuyHold_Return'] - df['Risk_Free']
)

df['SMA50_Excess'] = (
    df['SMA50_Return'] - df['Risk_Free']
)

df['SMA200_Excess'] = (
    df['SMA200_Return'] - df['Risk_Free']
)

df['Crossover_Excess'] = (
    df['Crossover_Return'] - df['Risk_Free']
)

df['Market_Excess'] = (
    df['Market_Return'] - df['Risk_Free']
)

print("Excess returns calculated.")


Excess returns calculated.


In [248]:
# ============================================================
# FINAL CAPM COMPARISON TABLE
# ============================================================

def calculate_capm_stats(data, excess_return_column):

    # Select the strategy and market excess returns
    regression_data = data[
        [excess_return_column, 'Market_Excess']
    ].dropna()

    print(regression_data.shape)

if regression_data.empty:
    print("No valid data available for CAPM regression")
    
    # Independent variable: Market Excess Return
    X = regression_data['Market_Excess']

    # Add intercept for Alpha
    X = sm.add_constant(X)

    # Dependent variable: Strategy Excess Return
    y = regression_data[excess_return_column]

    # Run OLS regression
    model = sm.OLS(y, X).fit()

    # Extract CAPM statistics
    alpha_daily = model.params['const']
    alpha_annual = alpha_daily * 252
    beta = model.params['Market_Excess']
    alpha_pvalue = model.pvalues['const']
    r_squared = model.rsquared

return {
        'Annualized Alpha': alpha_annual,
        'Alpha P-Value': alpha_pvalue,
        'Beta': beta,
        'R-Squared': r_squared,
        'Observations': len(regression_data)
     }


# ------------------------------------------------------------
# Calculate CAPM statistics for each strategy
# ------------------------------------------------------------

strategies = {
    'Buy & Hold': 'BuyHold_Excess',
    '50-Day SMA': 'SMA50_Excess',
    '200-Day SMA': 'SMA200_Excess',
    '50/200 SMA Crossover': 'Crossover_Excess'
}


# Create results list
results = []

for strategy_name, return_column in strategies.items():

    stats = calculate_capm_stats(
        df,
        return_column
    )

    stats['Strategy'] = strategy_name

    results.append(stats)


# Convert results into a dataframe
capm_table = pd.DataFrame(results)


# Put Strategy as the first column
capm_table = capm_table[
    [
        'Strategy',
        'Annualized Alpha',
        'Alpha P-Value',
        'Beta',
        'R-Squared',        
        'Observations'
    ]
]


# Format the table
capm_table['Annualized Alpha'] = (
    capm_table['Annualized Alpha'] * 100
)

capm_table['Annualized Alpha'] = (
    capm_table['Annualized Alpha'].round(2)
)

capm_table['Alpha P-Value'] = (
    capm_table['Alpha P-Value'].round(4)
)

capm_table['Beta'] = (
    capm_table['Beta'].round(4)
)

capm_table['R-Squared'] = (
    capm_table['R-Squared'].round(4)
)


    
# Display final table
print("============================================================")
print("FINAL CAPM STRATEGY COMPARISON")
print("============================================================")

display(capm_table)

NameError: name 'regression_data' is not defined

In [ ]:
# ============================================================
# FINAL CAPM ANALYSIS + ALPHA VISUALISATION
# ============================================================

# ------------------------------------------------------------
# 1. Make sure the strategy excess returns exist
# ------------------------------------------------------------

# Buy & Hold
df['BuyHold_Return'] = df['Asset_Return']
df['BuyHold_Excess'] = df['BuyHold_Return'] - df['Risk_Free']


# 50-Day SMA
df['SMA_50'] = (
    df['Asset_Price']
    .rolling(window=50)
    .mean()
    .shift(1)
)

df['Signal_50'] = np.where(
    df['Asset_Price'].shift(1) > df['SMA_50'],
    1,
    0
)

df['SMA50_Return'] = (
    df['Signal_50'] * df['Asset_Return']
)

df['SMA50_Excess'] = (
    df['SMA50_Return'] - df['Risk_Free']
)
    # 200-Day SMA
df['SMA_200'] = (
    df['Asset_Price']
    .rolling(window=200)
    .mean()
    .shift(1)
)

df['Signal_200'] = np.where(
    df['Asset_Price'].shift(1) > df['SMA_200'],
    1,
    0
)

df['SMA200_Return'] = (
    df['Signal_200'] * df['Asset_Return']
)

df['SMA200_Excess'] = (
    df['SMA200_Return'] - df['Risk_Free']
)


# 50/200 SMA Crossover
df['Crossover_Signal'] = np.where(
    df['SMA_50'] > df['SMA_200'],
    1,
    0
)

df['Crossover_Return'] = (
    df['Crossover_Signal'] * df['Asset_Return']
)

df['Crossover_Excess'] = (
    df['Crossover_Return'] - df['Risk_Free']
)
    # Market excess return
df['Market_Excess'] = (
    df['Market_Return'] - df['Risk_Free']
)


# ------------------------------------------------------------
# 2. Remove missing values
# ------------------------------------------------------------

df = df.dropna(
    subset=[
        'Market_Excess',
        'BuyHold_Excess',
        'SMA50_Excess',
        'SMA200_Excess',
        'Crossover_Excess'
    ]
)


# ------------------------------------------------------------
# 3. Create the two analysis periods
# ------------------------------------------------------------

# Two years before the most recent observation
split_date = df.index.max() - pd.DateOffset(years=2)

# Prior period
df_prior = df[df.index < split_date].copy()

# Recent 2-year period
df_recent = df[df.index >= split_date].copy()

# ------------------------------------------------------------
# 4. CAPM regression function
# ------------------------------------------------------------

def calculate_capm(data, strategy_column):

    regression_data = data[
        [strategy_column, 'Market_Excess']
    ].dropna()

    X = regression_data['Market_Excess']
    X = sm.add_constant(X)

    y = regression_data[strategy_column]

    model = sm.OLS(y, X).fit()

    # Daily Alpha
    alpha_daily = model.params['const']

    # Annualized Alpha
    alpha_annual = alpha_daily * 252

    # Beta
    beta = model.params['Market_Excess']

    # Alpha p-value
    alpha_pvalue = model.pvalues['const']

    # R-squared
    r_squared = model.rsquared

    return {
        'Annualized Alpha (%)': alpha_annual * 100,
        'Alpha P-Value': alpha_pvalue,
        'Beta': beta,
        'R-Squared': r_squared,
        'Observations': len(regression_data)
    }


# ------------------------------------------------------------
# 5. Define strategies
# ------------------------------------------------------------

strategies = {
    'Buy & Hold': 'BuyHold_Excess',
    '50-Day SMA': 'SMA50_Excess',
    '200-Day SMA': 'SMA200_Excess',
    '50/200 SMA Crossover': 'Crossover_Excess'
}


# ------------------------------------------------------------
# 6. Calculate CAPM results for both periods
# ------------------------------------------------------------

results = []

for period_name, period_data in [
    ('Prior Period', df_prior),
    ('Recent 2 Years', df_recent)
    ]:

    for strategy_name, strategy_column in strategies.items():

        stats = calculate_capm(
            period_data,
            strategy_column
        )

        results.append({
            'Period': period_name,
            'Strategy': strategy_name,
            **stats
        })


# ------------------------------------------------------------
# 7. Create final results table
# ------------------------------------------------------------

final_capm_table = pd.DataFrame(results)

# Round results
final_capm_table['Annualized Alpha (%)'] = (
    final_capm_table['Annualized Alpha (%)'].round(2)
)

final_capm_table['Alpha P-Value'] = (
    final_capm_table['Alpha P-Value'].round(4)
)

final_capm_table['Beta'] = (
    final_capm_table['Beta'].round(4)
)

for strategy_name, strategy_column in strategies.items():

        stats = calculate_capm(
            period_data,
            strategy_column
        )

        results.append({
            'Period': period_name,
            'Strategy': strategy_name,
            **stats
        })


# ------------------------------------------------------------
# 7. Create final results table
# ------------------------------------------------------------

final_capm_table = pd.DataFrame(results)

# Round results
final_capm_table['Annualized Alpha (%)'] = (
    final_capm_table['Annualized Alpha (%)'].round(2)
)

final_capm_table['Alpha P-Value'] = (
    final_capm_table['Alpha P-Value'].round(4)
)

final_capm_table['Beta'] = (
    final_capm_table['Beta'].round(4)
)

# ------------------------------------------------------------
# 10. Visualise Alpha
# ------------------------------------------------------------

import matplotlib.pyplot as plt

ax = alpha_plot_data.plot(
    kind='bar',
    figsize=(12, 6),
    width=0.75
)

plt.axhline(
    y=0,
    color='black',
    linewidth=1
)

plt.title(
    f'Annualized CAPM Alpha by Strategy — {asset_ticker}'
)

plt.xlabel('Strategy')
plt.ylabel('Annualized Alpha (%)')

plt.xticks(
    rotation=20,
    ha='right'
)

plt.legend(
    title='Period'
)

plt.grid(
    axis='y',
    linestyle='--',
    alpha=0.3
)

plt.tight_layout()
plt.show()
    

In [ ]:
# ============================================================
# CUMULATIVE RETURNS: ALL STRATEGIES VS MARKET
# ============================================================

import matplotlib.pyplot as plt

# Calculate cumulative returns for each strategy
cumulative_buyhold = (
    1 + df['BuyHold_Return']
).cumprod() - 1

cumulative_sma50 = (
    1 + df['SMA50_Return']
).cumprod() - 1

cumulative_sma200 = (
    1 + df['SMA200_Return']
).cumprod() - 1

cumulative_crossover = (
    1 + df['Crossover_Return']
).cumprod() - 1

# Calculate cumulative market return
cumulative_market = (
    1 + df['Market_Return']
).cumprod() - 1


# ============================================================
# PLOT
# ============================================================

plt.figure(figsize=(14, 7))

plt.plot(
    cumulative_buyhold.index,
    cumulative_buyhold * 100,
    label='Buy & Hold',
    linewidth=2
)

plt.plot(
    cumulative_sma50.index,
    cumulative_sma50 * 100,
    label='50-Day SMA',
    linewidth=2
)

plt.plot(
    cumulative_sma200.index,
    cumulative_sma200 * 100,
    label='200-Day SMA',
    linewidth=2
)

plt.plot(
    cumulative_crossover.index,
    cumulative_crossover * 100,
    label='50/200 SMA Crossover',
    linewidth=2
)

plt.plot(
    cumulative_market.index,
    cumulative_market * 100,
    label='S&P 500 Market',
    linewidth=2,
    linestyle='--'
)

# Zero-return reference line
plt.axhline(
    y=0,
    color='black',
    linewidth=1
)


# Chart formatting
plt.title(
    f'Cumulative Returns: {asset_ticker} Strategies vs S&P 500',
    fontsize=14
)

plt.xlabel('Date')
plt.ylabel('Cumulative Return (%)')

plt.legend()

plt.grid(
    True,
    linestyle='--',
    alpha=0.3
)

plt.tight_layout()
plt.show()

# ============================================================
# VOLATILITY CLUSTERING: ROLLING REALIZED VOLATILITY
# (aligned to same date range as cumulative returns chart above)
# ============================================================

df['RollingVol_21d'] = df['Asset_Return'].rolling(window=21).std() * np.sqrt(252) * 100

plt.figure(figsize=(14, 2))

plt.plot(
    df.index,
    df['RollingVol_21d'],
    linewidth=1,
    color='darkred'
)

# Shade a rough "elevated volatility" reference band so spikes stand out
plt.axhline(
    y=df['RollingVol_21d'].mean(),
    color='black',
    linewidth=1,
    linestyle='--',
    label=f"Mean Vol ({df['RollingVol_21d'].mean():.1f}%)"
)

# Optional: mark known major market events for easy visual cross-reference
event_dates = {
    'COVID Crash': '2020-03-01',
    '2022 Selloff': '2022-01-01',
}

for label, date_str in event_dates.items():
    event_date = pd.Timestamp(date_str)
    if event_date >= df.index.min() and event_date <= df.index.max():
        plt.axvline(x=event_date, color='gray', linewidth=1, linestyle=':', alpha=0.6)
        plt.text(
            event_date, plt.ylim()[1] * 0.95, label,
            rotation=90, verticalalignment='top', fontsize=8, color='gray'
        )

plt.title(f'{asset_ticker}: 21-Day Rolling Annualized Volatility', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Annualized Volatility (%)')

# Match x-axis limits to the chart above for direct visual comparison
plt.xlim(df.index.min(), df.index.max())

plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# FORMAL TEST: AUTOCORRELATION OF SQUARED RETURNS
# (confirms volatility clustering seen in the rolling vol chart)
# ============================================================

from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox

squared_returns = df['Asset_Return'] ** 2

fig, ax = plt.subplots(figsize=(14, 2))
plot_acf(squared_returns, lags=40, ax=ax)
ax.set_title(f'{asset_ticker}: ACF of Squared Returns (Volatility Clustering Check)')
plt.tight_layout()
plt.show()

# Ljung-Box test — null hypothesis is NO autocorrelation (i.e. no clustering)
lb_test = acorr_ljungbox(squared_returns, lags=[10, 20], return_df=True)
print("Ljung-Box test on squared returns:")
display(lb_test)
print("\nLow p-values (near 0) -> reject the null of no autocorrelation")
print("-> statistical confirmation of volatility clustering (ARCH effects).")